# IMPAC-D Teams API Demo

Admin
Instructor1
Student1
Student3

## Setting up the ENVIRONMENT


In [4]:
%%bash
set -euo pipefail

cat >/tmp/impacd_team_demo.env <<'EOF'
export LMS="http://local.openedx.io:8000" #8008
export COURSE_ID="course-v1:IITB+CS101+AU2026"
export TOPIC_ID="demo-teamset"
export TEAM_ID="impac-d-demo-team-479acfd92d4b4055895821fb0bf4615f"
export APP_ORIGIN="http://apps.local.openedx.io:1999"

export INSTRUCTOR_USERNAME="instructor1"
export STUDENT_USERNAME="student1"
export OTHER_USERNAME="student3"

export INSTRUCTOR_COOKIE_JAR="/tmp/openedx-instructor1-cookies.txt"
export STUDENT_COOKIE_JAR="/tmp/openedx-student1-cookies.txt"
EOF

cat >/tmp/impacd_team_demo_helpers.sh <<'EOF'
show_response() {
  local headers="$1"
  local body="$2"

  sed -e 's/\r$//' "$headers"
  echo
  if jq -e . "$body" >/dev/null 2>&1; then
    jq . "$body"
  else
    cat "$body"
    echo
  fi
}
EOF

cat /tmp/impacd_team_demo.env


export LMS="http://local.openedx.io:8000" #8008
export COURSE_ID="course-v1:IITB+CS101+AU2026"
export TOPIC_ID="demo-teamset"
export TEAM_ID="impac-d-demo-team-479acfd92d4b4055895821fb0bf4615f"
export APP_ORIGIN="http://apps.local.openedx.io:1999"

export INSTRUCTOR_USERNAME="instructor1"
export STUDENT_USERNAME="student1"
export OTHER_USERNAME="student3"

export INSTRUCTOR_COOKIE_JAR="/tmp/openedx-instructor1-cookies.txt"
export STUDENT_COOKIE_JAR="/tmp/openedx-student1-cookies.txt"


## Login / Refresh Cookies

Logs in both `instructor1` and `student1` using `password123`, matching the browser endpoint `/api/user/v2/account/login_session/`.

In [5]:
%%bash
set -euo pipefail
source /tmp/impacd_team_demo.env
source /tmp/impacd_team_demo_helpers.sh

login_user() {
  local username="$1"
  local cookie_jar="$2"
  local prefix="$3"
  local password="password123"
  local csrf body_file http_code

  body_file="$(mktemp)"
  trap 'rm -f "$body_file"' RETURN

  curl -sS -c "$cookie_jar" -b "$cookie_jar" "$LMS/api/user/v2/account/login_session/" >/dev/null

  csrf="$(awk '$6 == "csrftoken" { value = $7 } END { print value }' "$cookie_jar")"
  if [[ -z "$csrf" ]]; then
    echo "Could not get csrftoken for $username from login_session" >&2
    return 1
  fi

  http_code="$(
    curl -sS -o "$body_file" -w '%{http_code}' \
      -c "$cookie_jar" \
      -b "$cookie_jar" \
      -X POST "$LMS/api/user/v2/account/login_session/" \
      -H 'Accept: application/json, text/plain, */*' \
      -H 'Content-Type: application/x-www-form-urlencoded' \
      -H "Origin: $APP_ORIGIN" \
      -H "Referer: $APP_ORIGIN/" \
      -H "X-CSRFToken: $csrf" \
      --data-urlencode "email_or_username=$username" \
      --data-urlencode "next=/" \
      --data-urlencode "password=$password"
  )"

  if [[ "$http_code" != "200" ]] || ! grep -q '"success"[[:space:]]*:[[:space:]]*true' "$body_file"; then
    echo "Login failed for $username with HTTP $http_code" >&2
    cat "$body_file" >&2
    return 1
  fi

  csrf="$(awk '$6 == "csrftoken" { value = $7 } END { print value }' "$cookie_jar")"
  {
    printf 'export %s_CSRF=%q\n' "$prefix" "$csrf"
  } >>/tmp/impacd_team_demo.env

  echo "Logged in $username as $prefix"
}

login_user "$INSTRUCTOR_USERNAME" "$INSTRUCTOR_COOKIE_JAR" "INSTRUCTOR"
login_user "$STUDENT_USERNAME" "$STUDENT_COOKIE_JAR" "STUDENT"

grep -E '^(export (INSTRUCTOR|STUDENT)_CSRF=)' /tmp/impacd_team_demo.env


Logged in instructor1 as INSTRUCTOR
Logged in student1 as STUDENT
export INSTRUCTOR_CSRF=CIlMVPB7PAklgLIZeU0LTQ7UwVgH5vuU
export STUDENT_CSRF=gvfIlLGRYLEOAeR2hHRYneQraVrR8wq9


## Check Teamsets / Topics

If this returns zero results, add `demo-teamset` in Studio advanced settings before continuing.

In [17]:
%%bash
set -euo pipefail
source /tmp/impacd_team_demo.env
source /tmp/impacd_team_demo_helpers.sh

headers="$(mktemp)"
body="$(mktemp)"
trap 'rm -f "$headers" "$body"' EXIT

curl -sS -D "$headers" -o "$body" -b "$INSTRUCTOR_COOKIE_JAR" -G "$LMS/api/team/v0/topics/" \
  --data-urlencode "course_id=$COURSE_ID" \
  -H 'Accept: application/json' \
  -H 'USE-JWT-COOKIE: true'

show_response "$headers" "$body"


HTTP/1.1 200 OK
Date: Mon, 15 Jun 2026 08:55:48 GMT
Server: WSGIServer/0.2 CPython/3.11.8
Content-Type: application/json
Vary: Accept, Accept-Language, origin, Cookie
Allow: GET, HEAD, OPTIONS
X-Frame-Options: SAMEORIGIN
Content-Language: en
Content-Length: 247
Set-Cookie:  openedx-language-preference=""; Domain=local.openedx.io; expires=Thu, 01 Jan 1970 00:00:00 GMT; Max-Age=0; Path=/
Connection: close


{
  "next": null,
  "previous": null,
  "count": 1,
  "num_pages": 1,
  "current_page": 1,
  "start": 0,
  "results": [
    {
      "description": "Teamset for IMPAC-D demo",
      "name": "Demo Teamset",
      "id": "demo-teamset",
      "type": "open",
      "max_team_size": null,
      "team_count": 3
    }
  ],
  "sort_order": "name"
}


## Enroll Students In Course

Setup helper. Uses the instructor API to enroll usernames from `ENROLL_IDENTIFIERS` into `COURSE_ID`. Edit `ENROLL_IDENTIFIERS` as needed.

In [ ]:
%%bash
set -euo pipefail
source /tmp/impacd_team_demo.env
source /tmp/impacd_team_demo_helpers.sh

ENROLL_IDENTIFIERS="${ENROLL_IDENTIFIERS:-student3,student4}"
COURSE_ID_ENCODED="${COURSE_ID//+/%2B}"

headers="$(mktemp)"
body="$(mktemp)"
payload="$(mktemp)"
trap 'rm -f "$headers" "$body" "$payload"' EXIT

jq -n \
  --arg action "enroll" \
  --arg identifiers "$ENROLL_IDENTIFIERS" \
  --argjson auto_enroll false \
  --argjson email_students false \
  --arg reason "IMPAC-D team demo setup" \
  '{action:$action, identifiers:$identifiers, auto_enroll:$auto_enroll, email_students:$email_students, reason:$reason}' >"$payload"

curl -sS -D "$headers" -o "$body" -b "$INSTRUCTOR_COOKIE_JAR" -X POST "$LMS/courses/$COURSE_ID_ENCODED/instructor/api/students_update_enrollment" \
  -H 'Accept: application/json' \
  -H 'Content-Type: application/json' \
  -H "X-CSRFToken: $INSTRUCTOR_CSRF" \
  --data-binary "@$payload"

show_response "$headers" "$body"


## Show Enrolled Students

Uses the instructor analytics endpoint to show the active enrolled learners for `COURSE_ID`. Run this after enrollment setup to confirm the course roster.

In [18]:
%%bash
set -euo pipefail
source /tmp/impacd_team_demo.env

COURSE_ID_ENCODED="${COURSE_ID//+/%2B}"

headers="$(mktemp)"
body="$(mktemp)"
trap 'rm -f "$headers" "$body"' EXIT

curl -sS -D "$headers" -o "$body" -b "$INSTRUCTOR_COOKIE_JAR" -X POST "$LMS/courses/$COURSE_ID_ENCODED/instructor/api/get_students_features" \
  -H 'Accept: application/json' \
  -H "X-CSRFToken: $INSTRUCTOR_CSRF"

sed -e 's/\r$//' "$headers"
echo
jq '{course_id, students_count, students: [.students[] | {username, email, enrollment_mode, team}]}' "$body"


HTTP/1.1 403 Forbidden
Date: Mon, 15 Jun 2026 08:56:11 GMT
Server: WSGIServer/0.2 CPython/3.11.8
Content-Type: application/json
Vary: Accept, Accept-Language, origin, Cookie
Allow: POST, OPTIONS
Cache-Control: no-cache, no-store, must-revalidate
X-Frame-Options: SAMEORIGIN
Content-Language: en
Content-Length: 74
Set-Cookie:  openedx-language-preference=""; Domain=local.openedx.io; expires=Thu, 01 Jan 1970 00:00:00 GMT; Max-Age=0; Path=/
Set-Cookie:  lms_sessionid=1|29m5k5rkin2m9se8bmgo7pty7zusumnz|9PB7pH4ljoUm|ImFhMTJjMjljYmY3NTk5ZDE5ZTFhMzY5M2I4MWUzYTA3OTNiNTAxNGYyY2JmNmZlMjg0ZDc5YmYyZmVlYjA1ZGEi:1wZ375:i-n4bBma2wOYVyZGcdoIvY1USjw4DF4GXtdU7--P6G4; Domain=local.openedx.io; expires=Mon, 13 Jul 2026 08:56:11 GMT; HttpOnly; Max-Age=2419200; Path=/; SameSite=Lax
Connection: close




jq: error (at /tmp/tmp.j4vBbDLXDJ:0): Cannot iterate over null (null)


CalledProcessError: Command 'b'set -euo pipefail\nsource /tmp/impacd_team_demo.env\n\nCOURSE_ID_ENCODED="${COURSE_ID//+/%2B}"\n\nheaders="$(mktemp)"\nbody="$(mktemp)"\ntrap \'rm -f "$headers" "$body"\' EXIT\n\ncurl -sS -D "$headers" -o "$body" -b "$INSTRUCTOR_COOKIE_JAR" -X POST "$LMS/courses/$COURSE_ID_ENCODED/instructor/api/get_students_features" \\\n  -H \'Accept: application/json\' \\\n  -H "X-CSRFToken: $INSTRUCTOR_CSRF"\n\nsed -e \'s/\\r$//\' "$headers"\necho\njq \'{course_id, students_count, students: [.students[] | {username, email, enrollment_mode, team}]}\' "$body"\n'' returned non-zero exit status 5.

## Create Team

Run this only if `TEAM_ID` is empty, stale, full, or deleted. Copy the returned `id` into the setup cell or append it to `/tmp/impacd_team_demo.env`.

In [20]:
%%bash
set -euo pipefail
source /tmp/impacd_team_demo.env
source /tmp/impacd_team_demo_helpers.sh

headers="$(mktemp)"
body="$(mktemp)"
payload="$(mktemp)"
trap 'rm -f "$headers" "$body" "$payload"' EXIT

jq -n \
  --arg name "IMPAC-D Demo Team" \
  --arg course_id "$COURSE_ID" \
  --arg topic_id "$TOPIC_ID" \
  --arg description "Team used for IMPAC-D access-control demo" \
  --arg language "en" \
  '{name:$name, course_id:$course_id, topic_id:$topic_id, description:$description, language:$language}' >"$payload"

curl -sS -D "$headers" -o "$body" -b "$INSTRUCTOR_COOKIE_JAR" -X POST "$LMS/api/team/v0/teams/" \
  -H 'Accept: application/json' \
  -H 'Content-Type: application/json' \
  -H 'USE-JWT-COOKIE: true' \
  -H "X-CSRFToken: $INSTRUCTOR_CSRF" \
  --data-binary "@$payload"

show_response "$headers" "$body"


HTTP/1.1 200 OK
Date: Fri, 12 Jun 2026 08:38:00 GMT
Server: WSGIServer/0.2 CPython/3.11.8
Content-Type: application/json
Vary: Accept, Accept-Language, origin, Cookie
Allow: GET, POST, HEAD, OPTIONS
X-Frame-Options: SAMEORIGIN
Content-Language: en
Content-Length: 439
Set-Cookie:  openedx-language-preference=""; Domain=local.openedx.io; expires=Thu, 01 Jan 1970 00:00:00 GMT; Max-Age=0; Path=/


{
  "id": "impac-d-demo-team-60db5da389094f60a327c17138f92017",
  "discussion_topic_id": "60db5da389094f60a327c17138f92017",
  "name": "IMPAC-D Demo Team",
  "course_id": "course-v1:IITB+CS101+AU2026",
  "topic_id": "demo-teamset",
  "date_created": "2026-06-12T08:38:00.004412Z",
  "description": "Team used for IMPAC-D access-control demo",
  "country": "",
  "language": "en",
  "last_activity_at": "2026-06-12T08:38:00.004188Z",
  "membership": [],
  "organization_protected": false
}


## List Teams

In [19]:
%%bash
set -euo pipefail
source /tmp/impacd_team_demo.env
source /tmp/impacd_team_demo_helpers.sh

headers="$(mktemp)"
body="$(mktemp)"
trap 'rm -f "$headers" "$body"' EXIT

curl -sS -D "$headers" -o "$body" -b "$INSTRUCTOR_COOKIE_JAR" -G "$LMS/api/team/v0/teams/" \
  --data-urlencode "course_id=$COURSE_ID" \
  --data-urlencode "topic_id=$TOPIC_ID" \
  -H 'Accept: application/json' \
  -H 'USE-JWT-COOKIE: true'

show_response "$headers" "$body"


HTTP/1.1 200 OK
Date: Mon, 15 Jun 2026 08:56:21 GMT
Server: WSGIServer/0.2 CPython/3.11.8
Content-Type: application/json
Vary: Accept, Accept-Language, origin, Cookie
Allow: GET, POST, HEAD, OPTIONS
X-Frame-Options: SAMEORIGIN
Content-Language: en
Content-Length: 1432
Set-Cookie:  openedx-language-preference=""; Domain=local.openedx.io; expires=Thu, 01 Jan 1970 00:00:00 GMT; Max-Age=0; Path=/
Connection: close


{
  "next": null,
  "previous": null,
  "count": 3,
  "num_pages": 1,
  "current_page": 1,
  "start": 0,
  "results": [
    {
      "id": "impac-d-demo-team-479acfd92d4b4055895821fb0bf4615f",
      "discussion_topic_id": "479acfd92d4b4055895821fb0bf4615f",
      "name": "IMPAC-D Demo Team",
      "course_id": "course-v1:IITB+CS101+AU2026",
      "topic_id": "demo-teamset",
      "date_created": "2026-06-12T06:13:03.298721Z",
      "description": "Team used for IMPAC-D access-control demo",
      "country": "",
      "language": "en",
      "last_activity_at": "2026-06-12T06:13:

## Student Joins Self

Expected: `submit_team_membership_api=true`, legacy bypass fields true, membership created. If the student is already on a team, this may return the normal teamset duplicate error.

In [3]:
%%bash
set -euo pipefail
source /tmp/impacd_team_demo.env
source /tmp/impacd_team_demo_helpers.sh

headers="$(mktemp)"
body="$(mktemp)"
payload="$(mktemp)"
trap 'rm -f "$headers" "$body" "$payload"' EXIT

jq -n \
  --arg team_id "$TEAM_ID" \
  --arg username "$STUDENT_USERNAME" \
  '{team_id:$team_id, username:$username}' >"$payload"

curl -sS -D "$headers" -o "$body" -b "$STUDENT_COOKIE_JAR" -X POST "$LMS/api/team/v0/team_membership/?impacd_skip_legacy_access=1" \
  -H 'Accept: application/json' \
  -H 'Content-Type: application/json' \
  -H 'USE-JWT-COOKIE: true' \
  -H "X-CSRFToken: $STUDENT_CSRF" \
  --data-binary "@$payload"

show_response "$headers" "$body"


HTTP/1.1 502 Bad Gateway
Date: Tue, 16 Jun 2026 04:04:11 GMT
Server: WSGIServer/0.2 CPython/3.11.8
Content-Type: application/json
Vary: Cookie
Content-Length: 494
Connection: close


{
  "message": "External access-control error",
  "error": "Invalid access-control response: Access-control response is missing timestamp",
  "access_control_vars": {},
  "impacd_demo": {
    "external_access_control": "enforced",
    "legacy_access_checks_bypassed": true,
    "legacy_access_bypass_flag": "impacd_skip_legacy_access=1",
    "legacy_load_block_permission_bypassed": true,
    "legacy_permission_classes_bypassed": true,
    "legacy_team_access_checks_bypassed": true,
    "write_enforcement": "patched_model_save"
  }
}


## Student Adds Another Student

Expected: `submit_team_membership_api=false`, denied if student adds other student.

In [8]:
%%bash
set -euo pipefail
source /tmp/impacd_team_demo.env
source /tmp/impacd_team_demo_helpers.sh

headers="$(mktemp)"
body="$(mktemp)"
payload="$(mktemp)"
trap 'rm -f "$headers" "$body" "$payload"' EXIT

jq -n \
  --arg team_id "$TEAM_ID" \
  --arg username "$OTHER_USERNAME" \
  '{team_id:$team_id, username:$username}' >"$payload"

curl -sS -D "$headers" -o "$body" -b "$STUDENT_COOKIE_JAR" -X POST "$LMS/api/team/v0/team_membership/?impacd_skip_legacy_access=1" \
  -H 'Accept: application/json' \
  -H 'Content-Type: application/json' \
  -H 'USE-JWT-COOKIE: true' \
  -H "X-CSRFToken: $STUDENT_CSRF" \
  --data-binary "@$payload"

show_response "$headers" "$body"


HTTP/1.1 403 Forbidden
Date: Mon, 15 Jun 2026 08:21:47 GMT
Server: WSGIServer/0.2 CPython/3.11.8
Content-Type: application/json
Vary: Accept, Accept-Language, origin, Cookie
Allow: GET, POST, HEAD, OPTIONS
X-Frame-Options: SAMEORIGIN
Content-Language: en
X-IMPACD-Enforced: true
X-IMPACD-Legacy-Access-Bypassed: true
Content-Length: 560
Set-Cookie:  openedx-language-preference=""; Domain=local.openedx.io; expires=Thu, 01 Jan 1970 00:00:00 GMT; Max-Age=0; Path=/
Connection: close


{
  "error_msg": "Payload violates access-control constraints",
  "policies": [
    "mutation_constraints_0"
  ],
  "access_control_vars": {
    "submit_team_membership_api": true,
    "submit_xblock_handler_api": false,
    "view_dates_api": false,
    "view_grading_info_api": false
  },
  "impacd_demo": {
    "external_access_control": "enforced",
    "legacy_access_checks_bypassed": true,
    "legacy_access_bypass_flag": "impacd_skip_legacy_access=1",
    "legacy_load_block_permission_bypassed": true,
    "l

## Instructor Adds Student

Expected: allowed if instructor has staff access and the target user is eligible.

In [9]:
%%bash
set -euo pipefail
source /tmp/impacd_team_demo.env
source /tmp/impacd_team_demo_helpers.sh

headers="$(mktemp)"
body="$(mktemp)"
payload="$(mktemp)"
trap 'rm -f "$headers" "$body" "$payload"' EXIT

jq -n \
  --arg team_id "$TEAM_ID" \
  --arg username "$STUDENT_USERNAME" \
  '{team_id:$team_id, username:$username}' >"$payload"

curl -sS -D "$headers" -o "$body" -b "$INSTRUCTOR_COOKIE_JAR" -X POST "$LMS/api/team/v0/team_membership/?impacd_skip_legacy_access=1" \
  -H 'Accept: application/json' \
  -H 'Content-Type: application/json' \
  -H 'USE-JWT-COOKIE: true' \
  -H "X-CSRFToken: $INSTRUCTOR_CSRF" \
  --data-binary "@$payload"

show_response "$headers" "$body"


HTTP/1.1 500 Internal Server Error
Date: Mon, 15 Jun 2026 08:22:31 GMT
Server: WSGIServer/0.2 CPython/3.11.8
Content-Type: text/plain; charset=utf-8
X-Frame-Options: SAMEORIGIN
Vary: Accept-Language, origin, Cookie
Content-Language: en
X-IMPACD-Enforced: true
X-IMPACD-Legacy-Access-Bypassed: true
Content-Length: 144002
Set-Cookie:  openedx-language-preference=""; Domain=local.openedx.io; expires=Thu, 01 Jan 1970 00:00:00 GMT; Max-Age=0; Path=/
Connection: close


IntegrityError at /api/team/v0/team_membership/
(1062, "Duplicate entry '5-1' for key 'teams_courseteammembership.teams_courseteammembership_user_id_team_id_aa45a20c_uniq'")

Request Method: POST
Request URL: http://local.openedx.io:8000/api/team/v0/team_membership/?impacd_skip_legacy_access=1
Django Version: 5.2.11
Python Executable: /openedx/venv/bin/python
Python Version: 3.11.8
Python Path: ['/openedx/edx-platform', '', '/opt/pyenv/versions/3.11.8/lib/python311.zip', '/opt/pyenv/versions/3.11.8/lib/python3.11', '/opt/pyenv

## Remove User From Team

Cleanup helper. Uses the Teams detail endpoint: `DELETE /api/team/v0/team_membership/{team_id},{username}`. Set `REMOVE_USERNAME` before running if you want a different user.

In [ ]:
%%bash
set -euo pipefail
source /tmp/impacd_team_demo.env
source /tmp/impacd_team_demo_helpers.sh

REMOVE_USERNAME="${REMOVE_USERNAME:-$OTHER_USERNAME}"

headers="$(mktemp)"
body="$(mktemp)"
trap 'rm -f "$headers" "$body"' EXIT

curl -sS -D "$headers" -o "$body" -b "$INSTRUCTOR_COOKIE_JAR" -X DELETE "$LMS/api/team/v0/team_membership/$TEAM_ID,$REMOVE_USERNAME?admin=1" \
  -H 'Accept: application/json' \
  -H 'USE-JWT-COOKIE: true' \
  -H "X-CSRFToken: $INSTRUCTOR_CSRF"

show_response "$headers" "$body"
